In [14]:
from pathlib import Path
import math
import requests
from io import StringIO

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display
import os
import pathlib
import dataretrieval.waterdata as waterdata

DATA_DIR   = pathlib.Path().cwd().parent / 'data'   # adjust if running from a different directory
SWORD_GPKG = pathlib.Path().cwd().parent / 'data'   # na_sword_nodes_v17b_pnw.gpkg lives alongside the notebook


In [3]:
DATA_DIR

WindowsPath('/Users/masa6503/repos/swot-precip-validation/data')

In [29]:
match_distance_maximum=200
stations = pd.read_parquet(DATA_DIR /"dfms_v1_us_station_matches.parquet")
stations = stations.query(f"match_dist_m<{200}")
stations = stations[stations['usgs_site_no'].astype(str).str.len() == 8]
print(f"{len(stations):,} stations in DFMS.v1 with match_dist<200m")
stations.tail(10)

1,848 stations in DFMS.v1 with match_dist<200m


,usgs_site_no,node_id,reach_id,swot_obs,swot_orbit,match_dist_m,sword_width_m,sword_facc_km2,sword_lakeflag,sword_strm_order,sword_river_name,drain_area_km2,da_ratio,filter_path
2260,15564900,81220100420221,81220100421,5,164 265 293 442 470,86.996453,226.0,47405.496090,0,3,Koyukuk River,46593.92010,0.017418,distance_and_da
2263,15580095,81310200050881,81310200051,4,71 276 377 582,84.081914,84.0,1721.631536,0,1,Niukluk River,1831.12293,0.059795,distance_and_da
2265,15746900,81350100390341,81350100391,5,220 405 433 498 526,52.257224,48.0,518.158370,0,1,Wulik River,494.68809,0.047445,distance_and_da
2266,15747000,81350100360061,81350100361,4,220 248 405 433,55.053936,63.0,1889.524105,0,1,Wulik River,1825.94295,0.034821,distance_and_da
2267,15820000,81360000110921,81360000111,5,80 108 349 377 386,79.199592,63.0,3626.818962,0,1,NODATA,4402.98300,0.176281,distance_and_da
2268,15896000,81390200030541,81390200031,4,15 24 293 302 321,29.124972,42.0,8516.591890,0,2,NODATA,8650.56660,0.015487,distance_and_da
2269,15908000,81390400150641,81390400151,5,24 52 265 293 330,74.709554,126.0,5129.052091,0,2,Sagavanirktok River,4791.48150,0.070452,distance_and_da
2270,15910000,81390400150111,81390400151,5,24 52 265 293 330,70.294006,105.0,5348.125158,0,2,Sagavanirktok River,5697.97800,0.061399,distance_and_da
2271,15955000,81390800010251,81390800011,5,265 274 302 543 571,133.298937,113.5,4893.908002,0,2,Canning River,4998.68070,0.020960,distance_and_da
2272,15980000,81390900160651,81390900161,6,237 246 274 515 543 552 580,20.411896,36.0,1703.006427,0,1,Hulahula Riverr,1779.32313,0.042891,distance_and_da


In [30]:
usgs_codes = "USGS-"+np.array(stations.usgs_site_no)

In [33]:
list(usgs_codes)

['USGS-01010000',
 'USGS-01010500',
 'USGS-01014000',
 'USGS-01021500',
 'USGS-01029500',
 'USGS-01030000',
 'USGS-01030500',
 'USGS-01031500',
 'USGS-01033000',
 'USGS-01034500',
 'USGS-01036390',
 'USGS-01042500',
 'USGS-01046500',
 'USGS-01047000',
 'USGS-01047150',
 'USGS-01048000',
 'USGS-01049205',
 'USGS-01049265',
 'USGS-01054500',
 'USGS-01088400',
 'USGS-01092000',
 'USGS-01096550',
 'USGS-01098840',
 'USGS-01098860',
 'USGS-01131500',
 'USGS-01144500',
 'USGS-01154500',
 'USGS-01161280',
 'USGS-01170000',
 'USGS-01170006',
 'USGS-01170500',
 'USGS-01172000',
 'USGS-01172003',
 'USGS-01183999',
 'USGS-01190070',
 'USGS-01193000',
 'USGS-01200500',
 'USGS-01200600',
 'USGS-01312000',
 'USGS-01315000',
 'USGS-01315081',
 'USGS-01327750',
 'USGS-01328752',
 'USGS-01328770',
 'USGS-01329500',
 'USGS-01329550',
 'USGS-01329561',
 'USGS-01329650',
 'USGS-01331095',
 'USGS-01335505',
 'USGS-01335754',
 'USGS-01335755',
 'USGS-01335770',
 'USGS-01342602',
 'USGS-01342702',
 'USGS-013

In [39]:
locations = waterdata.get_monitoring_locations(monitoring_location_id=list(usgs_codes[0:100]), properties=['monitoring_location_name'])

In [51]:
timeseries = "2023-04-01T00:00:00Z/2026-03-30T12:31:12Z"
test = waterdata.get_latest_continuous(monitoring_location_id=list(usgs_codes[0:200]), parameter_code="00065", statistic_id="00011",
                                       time=timeseries)[0]

In [52]:
test

,geometry,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,latest_continuous_id
0,POINT (-73.66108 42.82483),9cee4a09f33b45ea863556c4dd06bb38,USGS-01335755,00065,00011,2026-02-28 19:00:00+00:00,NaN,ft,Provisional,[SEASONAL],2026-03-03 00:23:19.703807+00:00,a16174c6-2739-4cda-9c4d-e3d177c5e43c
1,POINT (-68.58528 47.28333),c289a526bea1493bb33ee6e8dd389b92,USGS-01014000,00065,00011,2026-03-03 02:30:00+00:00,4.95,ft,Provisional,None,2026-03-03 02:37:17.794271+00:00,cf4cbfea-0db7-4762-898b-c58e33c506f0
2,POINT (-69.31472 45.175),4a9c71c9a2c545c2a300e905d6e42f4e,USGS-01031500,00065,00011,2026-03-03 02:30:00+00:00,2.05,ft,Provisional,None,2026-03-03 02:47:14.192263+00:00,9d76ed77-dbea-45ab-bc27-4a71ec407ef7
3,POINT (-68.65139 45.23611),f9e0c9ff72a848b4af2aeac960ad59af,USGS-01034500,00065,00011,2026-03-03 02:30:00+00:00,3.31,ft,Provisional,None,2026-03-03 02:41:12.726023+00:00,58314b2e-3caa-4d47-80f2-80bcc4cd016d
4,POINT (-68.69667 44.82667),0addb5726065431caefd031624cacbd1,USGS-01036390,00065,00011,2026-03-03 02:30:00+00:00,NaN,ft,Provisional,[EQUIP],2026-03-03 02:40:15.121879+00:00,c7f22653-62a8-40fb-baef-cd6ee4c12e63
...,...,...,...,...,...,...,...,...,...,...,...,...
107,POINT (-80.69843 34.33208),16096cbe18494a2cb6eef44014940123,USGS-02147801,00065,00011,2026-03-03 03:15:00+00:00,10.92,ft,Provisional,None,2026-03-03 03:21:32.269802+00:00,c1aa007f-8dca-4f50-b796-ef4dce15b916
108,POINT (-81.42121 34.59514),3a173d4667394e3ea226ca3ac55611eb,USGS-02156500,00065,00011,2026-03-03 03:15:00+00:00,3.30,ft,Provisional,None,2026-03-03 03:28:17.514640+00:00,7d387806-2b74-492b-89a3-7aeeea36b020
109,POINT (-81.20954 34.05098),27db11808a564559806904384f75a869,USGS-02168504,00065,00011,2026-03-03 03:15:00+00:00,3.90,ft,Provisional,None,2026-03-03 03:29:17.519245+00:00,ec34a9d4-ddd6-4e09-9c80-5f968c525fc3
110,POINT (-73.93075 42.83053),bdc06b974c514f798263ec434017ca7a,USGS-01354500,00065,00011,2026-03-03 03:25:00+00:00,10.02,ft,Provisional,None,2026-03-03 03:31:16.100168+00:00,3364ed5c-59d7-4a14-8416-b5aff746c5c4


In [15]:
pcodes,metadata = waterdata.get_reference_table("parameter-codes")
# display(pcodes.head())

streamflow_pcodes = pcodes[pcodes['parameter_name'].str.contains('site|type', case=False, na=False)]
display(streamflow_pcodes[['parameter_code', 'parameter_name', 'parameter_description']])

,parameter_code,parameter_name,parameter_description
766,04119,Verticals in composite,"Verticals in composite sample, number"
3866,50276,Filter type,"Filter type, code"
3869,50280,Site visit purpose,"Site visit purpose, code"
13741,70001,Composite loctn in X-section,Composite location in cross section
14153,72326,Opposite direction,Opposite direction content quality assurance p...
15134,82923,"Atm. Dep. Type, wet","Atmospheric deposition type, wet, code"
15186,83205,"Atmos deposition type, bulk","Atmospheric deposition type, bulk, code"
15251,84164,Sampler type,"Sampler type, code"
15252,84171,Sample splitter type,"Sample splitter type, field, code"
15255,84174,POCIS sorbent type,"POCIS sorbent type, code"
